
### Notebook Overview

This notebook is designed to facilitate the collection and analysis of data needed to create metrics on the duration and stability of the main European and Middle Eastern governments. We'll be focusing on gathering relevant historical and political data to understand how long these governments tend to stay in power and how stable they are over time.



In [0]:
whogov_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load("/Volumes/geopolitics_data_catalog/raw/source_data/whogov_crosssectional_v4.0.csv")
)

In [0]:
whogov_df.limit(10).display()

In [0]:
result_df = (whogov_df
    .filter(
        whogov_df.country_name == "Italy"  # Filtro per il cognome
    )
    .select(
        "leader",    # Mantengo l'ID cliente
        "leader_start_date",     # Mantengo il nome
        "leader_end_date"       # Mantengo il cognome
    )
)

result_df.display()


In [0]:
result_df = result_df.dropDuplicates()

In [0]:
from pyspark.sql.functions import datediff, col, desc, when, current_date, expr

# Calcola durata governo in giorni, usando la data odierna se leader_end_date è null
# Usa try_cast per gestire valori 'NA' (diventano NULL invece di errore)
df_with_duration = result_df.withColumn(
    "duration_days", 
    datediff(
        when(expr("try_cast(leader_end_date as date)").isNull(), current_date()).otherwise(expr("try_cast(leader_end_date as date)")),
        expr("try_cast(leader_start_date as date)")
    )
)

df_with_duration.display()

In [0]:
longest_row = df_with_duration.orderBy(desc("duration_days")).limit(1).collect()[0]
longest_leader = longest_row["leader"]
longest_gov = longest_row["duration_days"]

In [0]:
longest_leader

In [0]:
# Creare schema processed se non esiste
spark.sql("CREATE SCHEMA IF NOT EXISTS geopolitics_data_catalog.processed")

# Filtrare record Moro 1963-1968 (1644 gg) - discrepante con Wikipedia che riporta
# i singoli governi Moro I-V con durata massima <1000 giorni ciascuno
df_filtered = df_with_duration.filter(
    ~((col("leader") == "Aldo Moro") & (col("leader_start_date") == "1963-12-05"))
)

# Salvare dati come tabella Unity Catalog
df_filtered.write \
    .mode("overwrite") \
    .saveAsTable("geopolitics_data_catalog.processed.italian_government_duration")

print("✅ Tabella salvata: geopolitics_data_catalog.processed.italian_government_duration")
print("ℹ️  Record Aldo Moro 1963-1968 escluso (discrepante con fonti storiche)")